# Hybrid Modeling: Neural Net, Boosters, Stacking, and β-Fusion

**Purpose.** Compare a neural network baseline, tree boosters, stacked boosters (NNLS meta-learner), and a hybrid fusion β·NN + (1−β)·booster on bundled insurance data. All results are computed from real train/validation/test splits with no row leakage.

## Learning objectives
- Build leakage-free train/validation/test and out-of-fold (OOF) prediction workflows.
- Fit stacking weights with `insurance_fl.fl.stacking.fit_stacking_meta` and optimize fusion β with `insurance_fl.fl.hybrid.optimize_beta`.
- Compare models with task-appropriate metrics and inspect β/loss behavior.
- Understand practical caveats: pooled holdout β optimization can conflict with federated no-raw-data-sharing assumptions.

## Background and math

### Stacking (level-1) with NNLS
Given base model OOF predictions $p^{(m)}$ on training rows, fit non-negative weights $\alpha$:
$$
\min_{\alpha \ge 0} \|y - \sum_m \alpha_m p^{(m)}\|_2^2 \quad (\text{NNLS}),
$$
then predict $\hat y_{stack}=\sum_m \alpha_m p^{(m)}(x)$.
*Meaning here:* combines complementary boosters using weights learned only from OOF predictions to reduce overfitting.

### Hybrid fusion (level-2)
$$
\hat y_{hyb} = \beta\, \hat y_{NN} + (1-\beta)\, \hat y_{booster}, \qquad \beta \in [0,1],
$$
with $\beta$ chosen to minimize validation loss. The project’s `optimize_beta` may return `beta_claim=None` if the hybrid does not beat a constant baseline (task marked uninformative).
*Meaning here:* blends neural and boosted predictors when beneficial; uninformative tasks should not claim a usable β.

### Leakage prevention
OOF predictions are generated so each training row is predicted by models not trained on it; β/meta-weights are fit on validation/OOF, and final metrics reported on a held-out test set with no overlap.
*Meaning here:* prevents optimistic bias from reusing training data for model selection/fusion.

## Assumptions and limitations
- Light mode uses subsampled data and small models for runtime; metrics are illustrative of the pipeline, not benchmarks.
- `hybrid.optimize_beta()` in the project is typically used on a global holdout. Pooling across countries conflicts with strict no-raw-data-sharing in federated settings; acceptable here for **synthetic** data only, or β should be derived from per-client held-out metrics in production FL.
- Optional boosters (xgboost/lightgbm/catboost) may be unavailable; sklearn fallbacks are used and reported.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
for _p in [_here, _here / "notebooks", _here.parent]:
    if (_p / "_nb_common.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
        break
import _nb_common as nb
nb.style()
rng = nb.set_seed()
nb.env_report()


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class Cfg:
    seed: int = nb.SEED
    n_rows: int = 800 if nb.MODE == "light" else 5000
    n_folds: int = 3 if nb.MODE == "light" else 5
    tasks = ("claim_risk_score", "is_high_risk")  # regression + binary
cfg = Cfg(); cfg


In [ ]:
import os, json, tempfile, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, roc_auc_score, roc_curve, precision_recall_curve, auc, accuracy_score, confusion_matrix, log_loss
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier, RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance

from insurance_fl.fl.stacking import fit_stacking_meta
from insurance_fl.fl.hybrid import optimize_beta

# optional boosters / NN backend (light mode forces stable sklearn fallbacks)
if nb.MODE == "light":
    xgb = lgbm = cat = torch = None
    print('[light mode] using sklearn fallbacks only for stability')
else:
    xgb = nb.optional('xgboost')
    lgbm = nb.optional('lightgbm')
    cat = nb.optional('catboost')
    torch = nb.optional('torch')
print('backends:', {'xgboost': bool(xgb), 'lightgbm': bool(lgbm), 'catboost': bool(cat), 'torch': bool(torch)})


In [ ]:
# Load and prepare data (pooled synthetic sample) with leakage-free split
clients = nb.load_clients(split='train', n_max=cfg.n_rows)
df = pd.concat([v.assign(client=k) for k,v in clients.items()], ignore_index=True)
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns]
feature_cols = [c for c in num_cols if c not in ['claim_risk_score','customer_lifetime_value','will_renew','is_high_risk','customer_segment']]
# single split: train/val/test
idx = np.arange(len(df))
idx_trainval, idx_test = train_test_split(idx, test_size=0.2, random_state=nb.SEED)
idx_train, idx_val = train_test_split(idx_trainval, test_size=0.25, random_state=nb.SEED)  # 0.25*0.8=0.2
assert set(idx_train).isdisjoint(idx_test)
assert set(idx_val).isdisjoint(idx_test)
assert set(idx_train).isdisjoint(idx_val)
print('sizes', len(idx_train), len(idx_val), len(idx_test))


In [ ]:
def task_meta(task):
    if task == 'claim_risk_score':
        return 'regression'
    if task == 'is_high_risk':
        return 'binary'
    raise ValueError(task)

# preprocessing
imp = SimpleImputer(strategy='median'); sc = StandardScaler()
X_all = imp.fit_transform(df[feature_cols].values)
X_all = sc.fit_transform(X_all)

def get_xy(task, indices):
    y = df.iloc[indices][task].values
    X = X_all[indices]
    if task_meta(task) == 'binary':
        y = y.astype(int)
    else:
        y = y.astype(float)
    return X, y

# simple NN (torch if available, else sklearn MLP) per task
def train_nn(task, Xtr, ytr, Xva, yva):
    ttype = task_meta(task)
    if torch is not None:
        import torch.nn as nn
        import torch.optim as optim
        d = Xtr.shape[1]
        if ttype == 'regression':
            model = nn.Sequential(nn.Linear(d,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1))
            loss_fn = nn.MSELoss()
        else:
            model = nn.Sequential(nn.Linear(d,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1))
            loss_fn = nn.BCEWithLogitsLoss()
        opt = optim.Adam(model.parameters(), lr=1e-3)
        Xt = torch.tensor(Xtr, dtype=torch.float32)
        yt = torch.tensor(ytr.reshape(-1,1), dtype=torch.float32)
        model.train()
        epochs = 20 if nb.MODE=='light' else 80
        for _ in range(epochs):
            opt.zero_grad(); out = model(Xt); loss = loss_fn(out, yt); loss.backward(); opt.step()
        model.eval()
        def pred_fn(X):
            with torch.no_grad():
                o = model(torch.tensor(X, dtype=torch.float32)).numpy().ravel()
            if ttype=='binary':
                return 1/(1+np.exp(-np.clip(o,-30,30)))
            return o
        return pred_fn, 'torch_mlp'
    else:
        if ttype == 'regression':
            from sklearn.neural_network import MLPRegressor
            m = MLPRegressor(hidden_layer_sizes=(64,32), max_iter=120 if nb.MODE=='light' else 300, random_state=nb.SEED)
        else:
            from sklearn.neural_network import MLPClassifier
            m = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=150 if nb.MODE=='light' else 400, random_state=nb.SEED)
        m.fit(Xtr, ytr)
        def pred_fn(X):
            if ttype=='binary':
                return m.predict_proba(X)[:,1]
            return m.predict(X)
        return pred_fn, 'sklearn_mlp'

# booster base learners with fallbacks
def booster_models(task):
    ttype = task_meta(task)
    models = {}
    if ttype == 'regression':
        if xgb is not None:
            models['xgb'] = xgb.XGBRegressor(n_estimators=40 if nb.MODE=='light' else 200, max_depth=3, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9, random_state=nb.SEED)
        else:
            models['xgb_fallback'] = GradientBoostingRegressor(random_state=nb.SEED)
        if lgbm is not None:
            models['lgbm'] = lgbm.LGBMRegressor(n_estimators=40 if nb.MODE=='light' else 200, max_depth=3, learning_rate=0.1, random_state=nb.SEED, verbose=-1)
        else:
            models['lgbm_fallback'] = RandomForestRegressor(n_estimators=50, random_state=nb.SEED)
        if cat is not None:
            models['cat'] = cat.CatBoostRegressor(iterations=40 if nb.MODE=='light' else 200, depth=4, learning_rate=0.1, random_seed=nb.SEED, verbose=False)
        else:
            models['cat_fallback'] = Ridge(alpha=1.0)
    else:
        if xgb is not None:
            models['xgb'] = xgb.XGBClassifier(n_estimators=40 if nb.MODE=='light' else 200, max_depth=3, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9, random_state=nb.SEED, eval_metric='logloss')
        else:
            models['xgb_fallback'] = GradientBoostingClassifier(random_state=nb.SEED)
        if lgbm is not None:
            models['lgbm'] = lgbm.LGBMClassifier(n_estimators=40 if nb.MODE=='light' else 200, max_depth=3, learning_rate=0.1, random_state=nb.SEED, verbose=-1)
        else:
            models['lgbm_fallback'] = RandomForestClassifier(n_estimators=50, random_state=nb.SEED)
        if cat is not None:
            models['cat'] = cat.CatBoostClassifier(iterations=40 if nb.MODE=='light' else 200, depth=4, learning_rate=0.1, random_seed=nb.SEED, verbose=False)
        else:
            models['cat_fallback'] = LogisticRegression(max_iter=500)
    return models

def fit_predict_model(name, model, task, Xtr, ytr, Xte):
    ttype = task_meta(task)
    model.fit(Xtr, ytr)
    if ttype=='binary':
        if hasattr(model,'predict_proba'):
            p = model.predict_proba(Xte)[:,1]
        else:
            p = model.predict(Xte).astype(float)
        return np.clip(p, 1e-6, 1-1e-6)
    return model.predict(Xte).astype(float)

print('feature count:', len(feature_cols))


In [ ]:
# Optional: demonstrate project booster training API in temp dir (best-effort)
proj_booster_info = {}
if nb.MODE == "full":
    try:
        from insurance_fl.fl.boosters import train_boosters_for_client
        with tempfile.TemporaryDirectory() as td:
            for task in cfg.tasks:
                Xtr, ytr = get_xy(task, idx_train)
                Xva, yva = get_xy(task, idx_val)
                res = train_boosters_for_client(
                    client_id='pooled_demo', x_train=Xtr, y_train={task:ytr}, x_val=Xva, y_val={task:yva},
                    output_dir=Path(td)/task, tasks=[task], n_folds=max(2,cfg.n_folds), seed=nb.SEED)
                proj_booster_info[task] = res.get(task, res)
        print('[project boosters] train_boosters_for_client executed in temp dir')
    except Exception as e:
        print('[project boosters] skipped/failed:', e)
else:
    print('[project boosters] skipped in light mode for runtime/memory; API signature checked separately')
proj_booster_info


## OOF stacking, NN training, and hybrid fusion (leakage-free)

In [ ]:
results = {}
all_preds = {}
for task in cfg.tasks:
    ttype = task_meta(task)
    Xtr, ytr = get_xy(task, idx_train)
    Xva, yva = get_xy(task, idx_val)
    Xte, yte = get_xy(task, idx_test)

    # base booster set (3 models) for stacking
    base = booster_models(task)
    base_names = list(base.keys())

    # OOF predictions on train only
    kf = KFold(n_splits=cfg.n_folds, shuffle=True, random_state=nb.SEED)
    oof = {name: np.zeros(len(Xtr), dtype=float) for name in base_names}
    for tr, va in kf.split(Xtr):
        for name, model in base.items():
            # fresh model per fold
            import copy
            m = copy.deepcopy(model)
            pred = fit_predict_model(name, m, task, Xtr[tr], ytr[tr], Xtr[va])
            oof[name][va] = np.asarray(pred, dtype=float).ravel()
    oof_list = [oof[n] for n in base_names]
    alpha = fit_stacking_meta(oof_list, ytr, fallback=True)
    alpha = np.asarray(alpha, dtype=float).ravel()

    # fit base models on full train and predict val/test
    base_val = {}; base_test = {}
    for name, model in base.items():
        import copy
        m = copy.deepcopy(model)
        base_val[name] = np.asarray(fit_predict_model(name, m, task, Xtr, ytr, Xva), dtype=float).ravel()
        base_test[name] = np.asarray(fit_predict_model(name, m, task, Xtr, ytr, Xte), dtype=float).ravel()

    def stack_pred(pred_dict):
        arrs = [pred_dict[n] for n in base_names]
        return sum(float(a) * p for a, p in zip(alpha, arrs))

    p_stack_val = stack_pred(base_val)
    p_stack_test = stack_pred(base_test)

    # NN
    nn_pred_fn, nn_name = train_nn(task, Xtr, ytr, Xva, yva)
    p_nn_val = np.asarray(nn_pred_fn(Xva), dtype=float).ravel()
    p_nn_test = np.asarray(nn_pred_fn(Xte), dtype=float).ravel()

    # constant baseline
    if ttype == 'regression':
        const = float(np.mean(ytr))
        p_const_val = np.full_like(yva, const, dtype=float)
        p_const_test = np.full_like(yte, const, dtype=float)
    else:
        const = float(np.clip(np.mean(ytr), 1e-6, 1-1e-6))
        p_const_val = np.full_like(yva, const, dtype=float)
        p_const_test = np.full_like(yte, const, dtype=float)

    # beta optimize on validation (global holdout in this notebook context)
    beta_claim, beta_diag = optimize_beta(p_nn_val, p_stack_val, yva, task_type=ttype, task_name=task)
    beta_used = float(beta_claim) if beta_claim is not None else float(beta_diag.get('beta', 0.0))
    task_status = beta_diag.get('task_status', 'INFORMATIVE')
    if task_status == 'UNINFORMATIVE' or beta_claim is None:
        p_hyb_test = beta_used * p_nn_test + (1 - beta_used) * p_stack_test  # diagnostic only
        hybrid_note = 'diagnostic (uninformative: no beta claim)'
    else:
        p_hyb_test = beta_used * p_nn_test + (1 - beta_used) * p_stack_test
        hybrid_note = 'usable'

    # metrics
    def metrics(y, p, ttype_):
        y = np.asarray(y).ravel(); p = np.asarray(p).ravel()
        if ttype_ == 'regression':
            return {'rmse': float(np.sqrt(mean_squared_error(y, p))), 'mae': float(np.mean(np.abs(y - p)))}
        else:
            pr = (p >= 0.5).astype(int)
            try:
                aucv = roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan
            except Exception:
                aucv = np.nan
            try:
                ll = log_loss(y, np.clip(p, 1e-6, 1-1e-6), labels=[0, 1])
            except Exception:
                ll = np.nan
            return {'accuracy': float(accuracy_score(y, pr)), 'auc': float(aucv) if np.isfinite(aucv) else np.nan, 'logloss': float(ll) if np.isfinite(ll) else np.nan}

    m = {
        'constant_baseline': metrics(yte, p_const_test, ttype),
        'nn': metrics(yte, p_nn_test, ttype),
        'best_single_booster_test': None,
        'stacked_boosters': metrics(yte, p_stack_test, ttype),
        'hybrid': metrics(yte, p_hyb_test, ttype),
    }
    # pick best single booster by validation metric
    if ttype == 'regression':
        best_name = min(base_names, key=lambda n: mean_squared_error(yva, base_val[n]))
    else:
        def val_score(n):
            try:
                return -roc_auc_score(yva, base_val[n])
            except Exception:
                return log_loss(yva, np.clip(base_val[n], 1e-6, 1-1e-6), labels=[0, 1])
        best_name = min(base_names, key=val_score)
    m['best_single_booster_test'] = {'model': best_name, **metrics(yte, base_test[best_name], ttype)}

    results[task] = {
        'task_type': ttype, 'alpha': alpha.tolist(), 'base_names': base_names,
        'beta_claim': beta_claim, 'beta_diag': beta_diag, 'hybrid_note': hybrid_note,
        'metrics': m, 'nn_name': nn_name,
    }
    all_preds[task] = {
        'yte': yte, 'yva': yva, 'Xte': Xte,
        'p_nn': p_nn_test, 'p_nn_val': p_nn_val,
        'p_stack': p_stack_test, 'p_stack_val': p_stack_val,
        'p_hyb': p_hyb_test, 'p_const': p_const_test,
        'base_test': base_test,
    }

print(json.dumps({k: {'beta_claim': v['beta_claim'], 'status': v['beta_diag'].get('task_status'), 'alpha': v['alpha']} for k, v in results.items()}, indent=2))


## Figures and diagnostics

In [ ]:
# Figure 1: model comparison bars
fig, axes = plt.subplots(1, len(cfg.tasks), figsize=(6.5*len(cfg.tasks),4))
if len(cfg.tasks)==1: axes=[axes]
for ax, task in zip(axes, cfg.tasks):
    r = results[task]; ttype = r['task_type']
    labels = ['constant','nn','best_booster','stacked','hybrid']
    if ttype=='regression':
        vals = [r['metrics']['constant_baseline']['rmse'], r['metrics']['nn']['rmse'], r['metrics']['best_single_booster_test']['rmse'], r['metrics']['stacked_boosters']['rmse'], r['metrics']['hybrid']['rmse']]
        ax.set_ylabel('RMSE (lower better)')
    else:
        vals = [r['metrics']['constant_baseline'].get('auc',np.nan), r['metrics']['nn'].get('auc',np.nan), r['metrics']['best_single_booster_test'].get('auc',np.nan), r['metrics']['stacked_boosters'].get('auc',np.nan), r['metrics']['hybrid'].get('auc',np.nan)]
        ax.set_ylabel('AUC (higher better)')
    ax.bar(labels, vals, color=['#888888','#2a6f97','#edae49','#00798c','#d1495b'])
    ax.set_title(f'{task} ({ttype})\nstatus={r["beta_diag"].get("task_status")}')
    ax.tick_params(axis='x', rotation=20)
fig.tight_layout(); nb.save_fig(fig, 'hybrid_model_comparison'); plt.show()


**How to read this / what it shows.** Test-set comparison across baselines and ensembles. For regression lower RMSE is better; for classification higher AUC is better. Hybrid bars marked uninformative in results should be interpreted cautiously.

In [ ]:
# Figure 2: beta vs loss curves (validation)
fig, axes = plt.subplots(1, len(cfg.tasks), figsize=(6 * len(cfg.tasks), 4))
if len(cfg.tasks) == 1:
    axes = [axes]
betas = np.linspace(0, 1, 41)
for ax, task in zip(axes, cfg.tasks):
    ttype = results[task]['task_type']
    p = all_preds[task]
    p_nn_v = np.asarray(p['p_nn_val'], dtype=float).ravel()
    p_stack_v = np.asarray(p['p_stack_val'], dtype=float).ravel()
    yva = np.asarray(p['yva']).ravel()
    losses = []
    for b in betas:
        hyb = b * p_nn_v + (1 - b) * p_stack_v
        if ttype == 'regression':
            losses.append(mean_squared_error(yva, hyb))
        else:
            losses.append(log_loss(yva, np.clip(hyb, 1e-6, 1-1e-6), labels=[0, 1]))
    ax.plot(betas, losses, label='hybrid loss')
    bstar = results[task]['beta_diag'].get('beta', None)
    if bstar is not None:
        ax.axvline(bstar, color='red', ls='--', label=f"beta*={bstar:.3f}")
    ax.set_title(f'Beta vs validation loss: {task}')
    ax.set_xlabel('beta (NN weight)')
    ax.set_ylabel('loss')
    ax.legend()
fig.tight_layout(); nb.save_fig(fig, 'hybrid_beta_curve'); plt.show()


**How to read this / what it shows.** Validation loss as β sweeps from booster-only (0) to NN-only (1). The selected β is the optimizer/argmin choice; if the task is uninformative, β is not a usable claim.

In [ ]:
# Figure 3: residuals vs predicted (regression) and ROC/PR (binary)
fig, axes = plt.subplots(1, len(cfg.tasks), figsize=(6*len(cfg.tasks),4))
if len(cfg.tasks)==1: axes=[axes]
for ax, task in zip(axes, cfg.tasks):
    ttype = results[task]['task_type']
    p = all_preds[task]; yte = p['yte']
    if ttype=='regression':
        pred = p['p_hyb']
        resid = yte - pred
        ax.scatter(pred, resid, s=10, alpha=0.4)
        ax.axhline(0, color='k', lw=1)
        ax.set_title(f'Residuals vs predicted (hybrid): {task}'); ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
    else:
        pred = p['p_hyb']
        fpr,tpr,_ = roc_curve(yte,pred)
        ax.plot(fpr,tpr,label=f"ROC AUC={results[task]['metrics']['hybrid'].get('auc',np.nan):.3f}")
        prec,rec,_ = precision_recall_curve(yte,pred)
        ax.plot(rec,prec,label=f"PR curve")
        ax.plot([0,1],[0,1],'k--',lw=1)
        ax.set_title(f'ROC/PR (hybrid): {task}'); ax.set_xlabel('FPR or Recall'); ax.set_ylabel('TPR or Precision'); ax.legend()
fig.tight_layout(); nb.save_fig(fig, 'hybrid_residuals_rocpr'); plt.show()


**How to read this / what it shows.** Regression: residual patterns indicate bias/heteroscedasticity. Classification: ROC/PR summarize discrimination and precision-recall behavior on the test split.

In [ ]:
# Figure 4: stacking weights and permutation importance (best single booster)
fig, axes = plt.subplots(1, len(cfg.tasks), figsize=(6.5*len(cfg.tasks),4))
if len(cfg.tasks)==1: axes=[axes]
for ax, task in zip(axes, cfg.tasks):
    r = results[task]
    names = r['base_names']; alpha = r['alpha']
    x = np.arange(len(names))
    ax.bar(x, alpha, color='#00798c')
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=20)
    ax.set_title(f'Stacking NNLS weights: {task}'); ax.set_ylabel('Weight')
fig.tight_layout(); nb.save_fig(fig, 'hybrid_stacking_weights'); plt.show()

# Permutation importance for a representative fitted model (retrain best booster on train)
fig, axes = plt.subplots(1, len(cfg.tasks), figsize=(7*len(cfg.tasks),4))
if len(cfg.tasks)==1: axes=[axes]
for ax, task in zip(axes, cfg.tasks):
    Xtr,ytr = get_xy(task, idx_train); Xte,yte = get_xy(task, idx_test)
    best = results[task]['metrics']['best_single_booster_test']['model']
    import copy
    m = copy.deepcopy(booster_models(task)[best])
    m.fit(Xtr,ytr)
    try:
        scoring = 'neg_mean_squared_error' if results[task]['task_type']=='regression' else 'roc_auc'
        Xpi, ypi = (Xte, yte) if nb.MODE!='light' or len(yte)<=200 else (Xte[:200], yte[:200])
        pi = permutation_importance(m, Xpi, ypi, n_repeats=2 if nb.MODE=='light' else 10, random_state=nb.SEED, scoring=scoring)
        imp_idx = np.argsort(pi.importances_mean)[-10:]
        ax.barh(np.array(feature_cols)[imp_idx], pi.importances_mean[imp_idx])
        ax.set_title(f'Permutation importance: {task} ({best})')
    except Exception as e:
        ax.text(0.5,0.5,f'importance skipped: {e}', ha='center')
fig.tight_layout(); nb.save_fig(fig, 'hybrid_feature_importance'); plt.show()


**How to read this / what it shows.** Left panels: NNLS stacking weights over base boosters. Right panels: permutation importance on the test split for the selected booster (model-agnostic; higher means larger performance drop when shuffled).

In [ ]:
# Figure 5: calibration/confusion for binary task if present
bin_tasks = [t for t in cfg.tasks if results[t]['task_type']=='binary']
if bin_tasks:
    t = bin_tasks[0]
    yte = all_preds[t]['yte']; p = all_preds[t]['p_hyb']
    pr = (p>=0.5).astype(int)
    cm = confusion_matrix(yte, pr, labels=[0,1])
    fig, axes = plt.subplots(1,2, figsize=(10,4))
    im = axes[0].imshow(cm, cmap='Blues')
    axes[0].set_title(f'Confusion matrix (hybrid): {t}')
    axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1]); axes[0].set_xlabel('Pred'); axes[0].set_ylabel('True')
    for i in range(2):
        for j in range(2): axes[0].text(j,i,str(cm[i,j]),ha='center',va='center')
    # calibration
    from sklearn.calibration import calibration_curve
    try:
        pt, pp = calibration_curve(yte, p, n_bins=min(10, max(3,len(yte)//30)), strategy='quantile')
        axes[1].plot(pp,pt,'o-',label='hybrid'); axes[1].plot([0,1],[0,1],'k--')
        axes[1].set_title('Calibration curve'); axes[1].set_xlabel('Mean predicted prob'); axes[1].set_ylabel('Fraction positive')
    except Exception as e:
        axes[1].text(0.5,0.5,str(e),ha='center')
    fig.tight_layout(); nb.save_fig(fig, 'hybrid_calibration_confusion'); plt.show()
else:
    print('[skip] no binary task for calibration/confusion')


**How to read this / what it shows.** Confusion matrix shows classification errors at threshold 0.5; calibration compares predicted probabilities to observed outcomes.

## Quantitative tables


In [ ]:
rows=[]
for task,r in results.items():
    for model_name, met in r['metrics'].items():
        if isinstance(met, dict) and 'model' in met and len(met)>1:
            # best single booster
            row={'task':task,'model':f"best_single_booster({met['model']})"}
            row.update({k:v for k,v in met.items() if k!='model'})
            rows.append(row)
        elif isinstance(met, dict):
            row={'task':task,'model':model_name}
            row.update(met)
            rows.append(row)
comp_df = pd.DataFrame(rows)
comp_df


In [ ]:
beta_df = pd.DataFrame([{ 'task':t, 'task_status':r['beta_diag'].get('task_status'), 'beta_claim':r['beta_claim'], 'beta_raw':r['beta_diag'].get('beta'), 'usable_beta':r['beta_diag'].get('usable_beta'), 'loss_baseline':r['beta_diag'].get('loss_baseline'), 'loss_hybrid':r['beta_diag'].get('loss_after_fusion'), 'beats_baseline':r['beta_diag'].get('beats_baseline'), 'note':'optimize_beta used global pooled validation holdout (synthetic demo); in federated no-raw-data-sharing settings derive beta from per-client held-out metrics' } for t,r in results.items()])
beta_df


## Sensitivity / ablation
- **Base learner set:** swapping/adding boosters changes stacking weights and hybrid gains.
- **β selection:** depends on validation split and task informativeness; `optimize_beta` can declare uninformative tasks.
- **Data split/seed:** OOF and test metrics vary with sampling in light mode.

## Limitations and failure cases
- Pooled global holdout for β conflicts with strict federated privacy constraints; acceptable only for synthetic data in this demo.
- Small/light-mode samples and few epochs can make NN/booster comparisons noisy.
- If base learners are highly correlated, stacking gains may be limited.

## Conclusions and further experiments
- The notebook computes NN, booster, stacked, and hybrid results with leakage-free splits using project stacking/hybrid utilities.
- β and stacking weights provide interpretable fusion diagnostics, subject to the stated caveats.
- Further work: per-client booster training with federated-compatible β selection, hyperparameter tuning, and SHAP-based explanations when installed.

## References
- D. H. Wolpert. "Stacked Generalization." *Neural Networks*, 1992.
- T. Chen and C. Guestrin. "XGBoost: A Scalable Tree Boosting System." *KDD*, 2016.
- G. Ke et al. "LightGBM: A Highly Efficient Gradient Boosting Decision Tree." *NeurIPS*, 2017.
